In [ ]:
from datasets import load_dataset

chinese_dataset = load_dataset("json", data_files={
    "train": "../data/amazon_reviews_multi_zh_raw/zh/train.jsonl.gz",
    "validation": "../data/amazon_reviews_multi_zh_raw/zh/validation.jsonl.gz",
    "test": "../data/amazon_reviews_multi_zh_raw/zh/test.jsonl.gz",
})

english_dataset = load_dataset("json", data_files={
    "train": "../data/amazon_reviews_multi_en_raw/en/train.jsonl.gz",
    "validation": "../data/amazon_reviews_multi_en_raw/en/validation.jsonl.gz",
    "test": "../data/amazon_reviews_multi_en_raw/en/test.jsonl.gz",
})


chinese_dataset

In [ ]:
def filter_books(example):
    return (
        example["product_category"] == "book"
        or example["product_category"] == "digital_ebook_purchase"
    )
chinese_books = chinese_dataset.filter(filter_books)
english_books = english_dataset.filter(filter_books)

In [ ]:
from datasets import concatenate_datasets, DatasetDict

books_dataset = DatasetDict()

for split in english_books.keys():
    books_dataset[split] = concatenate_datasets(
        [english_books[split], chinese_books[split]]
    )
    books_dataset[split] = books_dataset[split].shuffle(seed=42)

In [ ]:
books_dataset = books_dataset.filter(
    lambda x: x["review_title"] is not None and (
        len(x["review_title"]) > 5 if x["language"] == "zh" 
        else len(x["review_title"].split()) > 2
    )
)


In [ ]:
from transformers import AutoTokenizer
model_checkpoint = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
max_input_length = 512
max_target_length = 30

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["review_body"],
        max_length=max_input_length,
        truncation=True,
    )
    labels = tokenizer(
        examples["review_title"], max_length=max_target_length, truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_datasets = books_dataset.map(preprocess_function, batched=True)

In [ ]:
import evaluate
rouge_score = evaluate.load("rouge")

In [ ]:
import re
from nltk.tokenize import sent_tokenize

def three_sentence_summary(text, lang):
    if lang == "zh":
        sents = re.split(r"[。！？；]|\.{3}", text)  # 中文句子分割
    else:
        sents = sent_tokenize(text)
    return "\n".join([s.strip() for s in sents if s.strip()][:3])

In [ ]:
def evaluate_baseline(dataset, metric):
    summaries = [three_sentence_summary(text, lang) 
                 for text, lang in zip(dataset["review_body"], dataset["language"])]
    return metric.compute(predictions=summaries, references=dataset["review_title"])

In [ ]:
import pandas as pd

score = evaluate_baseline(books_dataset["validation"], rouge_score)
rouge_dict = {}
for rn in ["rouge1", "rouge2", "rougeL", "rougeLsum"]:
    rouge_dict[rn] = round(score[rn] * 100, 2)
rouge_dict



In [ ]:
from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

In [ ]:
from transformers import Seq2SeqTrainingArguments
batch_size = 8
num_train_epochs = 3
# 每个训练周期都输出训练损失
logging_steps = len(tokenized_datasets["train"]) // batch_size
model_name = model_checkpoint.split("/")[-1]

args = Seq2SeqTrainingArguments(
    output_dir=f"{model_name}-finetuned-amazon-en-zh",
    eval_strategy="epoch",
    learning_rate=5.6e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,
    logging_steps=logging_steps,
    push_to_hub=True,
)

In [ ]:
def split_sents(text):
    """中英文分句"""
    if re.search(r'[\u4e00-\u9fff]', text):  # 含中文
        sents = re.split(r"[。！？；]|\.{3}", text)
    else:
        sents = sent_tokenize(text)
    return [s.strip() for s in sents if s.strip()]

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds = ["\n".join(split_sents(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(split_sents(label.strip())) for label in decoded_labels]
    
    result = rouge_score.compute(
        predictions=decoded_preds, references=decoded_labels, use_stemmer=True
    )
    return {k: round(v * 100, 4) for k, v in result.items()}

In [ ]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(
    books_dataset["train"].column_names
)

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
from transformers import pipeline
hub_model_id = "Kate-lf/mt5-small-finetuned-amazon-en-zh"
summarizer = pipeline("summarization", model=hub_model_id)

In [ ]:
def print_summary(idx):
    review = books_dataset["test"][idx]["review_body"]
    title = books_dataset["test"][idx]["review_title"]
    summary = summarizer(books_dataset["test"][idx]["review_body"])[0]["summary_text"]
    print(f"'>>> Review: {review}'")
    print(f"\n'>>> Title: {title}'")
    print(f"\n'>>> Summary: {summary}'")